# 🧠 Tiny LLM Q&A Memorization PoC

## Objective
Validate training a tiny instruction-tuned model (~1-4MB) for Q&A memorization on ESP32-S3.

## 📦 Setup & Dependencies

In [116]:
!git clone https://github.com/karpathy/llama2.c.git
%cd llama2.c
!pip install -q torch numpy sentencepiece tqdm

Cloning into 'llama2.c'...
remote: Enumerating objects: 1517, done.
remote: Total 1517 (delta 0), reused 0 (delta 0), pack-reused 1517 (from 1)
Receiving objects: 100% (1517/1517), 1.43 MiB | 28.12 MiB/s, done.
Resolving deltas: 100% (930/930), done.
/content/llama2.c/llama2.c/llama2.c/llama2.c/llama2.c/llama2.c/llama2.c/llama2.c/llama2.c/llama2.c/llama2.c


In [117]:
import os
import random
import torch
import numpy as np

random.seed(42)
os.makedirs('qa_data', exist_ok=True)
os.makedirs('qa_output', exist_ok=True)
print('✅ Setup complete!')

✅ Setup complete!


## 🎨 Generate Synthetic Q&A Dataset

In [118]:
# Sample profile with factoids
FACTOIDS = [
    ('Name', 'Alphons Jaimon'),
    ('Email', 'chat@ajv009.com'),
    ('Current Job', 'GenAI Engineer at Etherwise'),
    ('Previous Job', 'AI Engineer at QED42'),
    ('Python Skill', 'Knows Python'),
    ('PHP Skill', 'Knows PHP'),
    ('Rust Skill', 'Knows Rust'),
    ('JavaScript Skill', 'Knows JavaScript'),
]

# Generate Q&A pairs with more variations
qa_pairs = []
for fact_type, answer in FACTOIDS:
    if 'Name' in fact_type:
        qa_pairs.append(('What is the name?', answer))
        qa_pairs.append(('Who is this person?', answer))
        qa_pairs.append(('Tell me the name.', answer))
    elif 'Email' in fact_type:
        qa_pairs.append(('What is the email?', answer))
        qa_pairs.append(('How to contact?', answer))
        qa_pairs.append(('Email address?', answer))
    elif 'Current Job' in fact_type:
        qa_pairs.append(('Where does the person work?', answer))
        qa_pairs.append(('What is the current job?', answer))
        qa_pairs.append(('Current employer?', answer))
    elif 'Previous Job' in fact_type:
        qa_pairs.append(('Where did the person work before?', answer))
        qa_pairs.append(('Previous employer?', answer))
    elif 'Skill' in fact_type:
        skill = answer.split()[-1]
        qa_pairs.append((f'Does the person know {skill}?', f'Yes, {answer.lower()}.'))
        qa_pairs.append((f'What about {skill} skills?', answer))

# Add general questions for variety
qa_pairs.extend([
    ('Who is Alphons?', 'Alphons Jaimon is a GenAI Engineer at Etherwise.'),
    ('What does Alphons do?', 'Alphons works as a GenAI Engineer.'),
    ('Tell me about Alphons.', 'Alphons Jaimon works at Etherwise and knows Python, PHP, Rust, and JavaScript.'),
    ('Where can I reach Alphons?', 'You can reach Alphons at chat@ajv009.com.'),
])

print(f'Generated {len(qa_pairs)} Q&A pairs')
for q, a in qa_pairs[:3]:
    print(f'Q: {q}')
    print(f'A: {a}')
    print()

Generated 23 Q&A pairs
Q: What is the name?
A: Alphons Jaimon

Q: Who is this person?
A: Alphons Jaimon

Q: Tell me the name.
A: Alphons Jaimon



In [119]:
# Format and save dataset with instruction-tuning format
# Format: <|user|>question<|assistant|>answer<|end|>
formatted = [f'<|user|>{q}<|assistant|>{a}<|end|>' for q, a in qa_pairs]
formatted_repeated = formatted * 5  # 5x repetition for memorization
random.shuffle(formatted_repeated)

split_idx = int(len(formatted_repeated) * 0.9)
train_data = formatted_repeated[:split_idx]
val_data = formatted_repeated[split_idx:]

with open('qa_data/train.txt', 'w') as f:
    f.write('\n'.join(train_data))
with open('qa_data/val.txt', 'w') as f:
    f.write('\n'.join(val_data))

print(f'Saved {len(train_data)} train, {len(val_data)} val samples')
print(f'Example: {train_data[0][:100]}...')

Saved 103 train, 12 val samples
Example: <|user|>Tell me about Alphons.<|assistant|>Alphons Jaimon works at Etherwise and knows Python, PHP, ...


## 🔤 Train Custom Tokenizer

In [120]:
!pip install -q sentencepiece
import sentencepiece as spm

# Prepare corpus
with open('qa_data/tokenizer_corpus.txt', 'w') as f:
    f.write('\n'.join(train_data + val_data))

# Train tokenizer - let BPE learn special tokens naturally
spm.SentencePieceTrainer.train(
    input='qa_data/tokenizer_corpus.txt',
    model_prefix='qa_data/qa_tok',
    vocab_size=227,  # Reduced to match corpus
    model_type='bpe'
)
print('✅ Tokenizer trained (special tokens learned naturally)')

✅ Tokenizer trained (special tokens learned naturally)


In [121]:
# Test tokenizer
sp = spm.SentencePieceProcessor()
sp.load('qa_data/qa_tok.model')
test = '<|user|>What is the name?<|assistant|>Alphons Jaimon<|end|>'
tokens = sp.encode(test)
print(f'Test: {test}')
print(f'Tokens: {tokens}')
print(f'Decoded: {sp.decode(tokens)}')

# Check how special tokens are encoded
user_tokens = sp.encode('<|user|>')
assistant_tokens = sp.encode('<|assistant|>')
end_tokens = sp.encode('<|end|>')
print(f'\n<|user|> tokens: {user_tokens}')
print(f'<|assistant|> tokens: {assistant_tokens}')
print(f'<|end|> tokens: {end_tokens}')

Test: <|user|>What is the name?<|assistant|>Alphons Jaimon<|end|>
Tokens: [16, 17, 4, 44, 59, 35, 146, 21, 19, 4, 33, 80, 3, 14, 4]
Decoded: <|user|>What is the name?<|assistant|>Alphons Jaimon<|end|>

<|user|> tokens: [16, 17, 4]
<|assistant|> tokens: [16, 19, 4]
<|end|> tokens: [16, 14, 4]


In [122]:
!python tokenizer.py --tokenizer-model=qa_data/qa_tok.model

## 🏗️ Prepare Training Data

In [123]:
import numpy as np

def tokenize_file(input_file, output_file, tokenizer):
    with open(input_file) as f:
        text = f.read()
    tokens = np.array(tokenizer.encode(text), dtype=np.uint16)
    with open(output_file, 'wb') as f:
        f.write(tokens.tobytes())
    return len(tokens)

train_tokens = tokenize_file('qa_data/train.txt', 'qa_data/train.bin', sp)
val_tokens = tokenize_file('qa_data/val.txt', 'qa_data/val.bin', sp)
print(f'Train: {train_tokens:,} tokens, Val: {val_tokens:,} tokens')

Train: 1,930 tokens, Val: 210 tokens


## 🎓 Train Model

In [124]:
from torch.utils.data import Dataset, DataLoader

class QADataset(Dataset):
    def __init__(self, path, seq_len=256):
        self.seq_len = seq_len
        self.tokens = np.fromfile(path, dtype=np.uint16).astype(np.int32)
    
    def __len__(self):
        return max(1, len(self.tokens) // self.seq_len)
    
    def __getitem__(self, idx):
        start = idx * self.seq_len
        chunk = self.tokens[start:start+self.seq_len+1]
        if len(chunk) < self.seq_len + 1:
            chunk = np.pad(chunk, (0, self.seq_len+1-len(chunk)))
        x = torch.from_numpy(chunk[:-1].astype(np.int64))
        y = torch.from_numpy(chunk[1:].astype(np.int64))
        return x, y

train_ds = QADataset('qa_data/train.bin')
val_ds = QADataset('qa_data/val.bin')
print(f'Datasets ready: {len(train_ds)} train, {len(val_ds)} val batches')

Datasets ready: 7 train, 1 val batches


In [125]:
from model import Transformer, ModelArgs

model_args = ModelArgs(
    dim=128,
    n_layers=4,
    n_heads=8,
    n_kv_heads=4,
    vocab_size=227,  # Must match tokenizer,
    max_seq_len=256,
    dropout=0.1
)

model = Transformer(model_args)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} params ({n_params/1e6:.2f}M)')
print(f'Expected int8 size: {n_params/1024/1024:.2f} MB')

Model: 1,013,248 params (1.01M)
Expected int8 size: 0.97 MB


In [126]:
# Training setup
from tqdm import tqdm
import time

batch_size = 8
learning_rate = 5e-4
max_iters = 500  # Quick PoC

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)
optimizer = model.configure_optimizers(0.1, learning_rate, (0.9, 0.95), device)

print(f'Training on {device} for {max_iters} iterations')

num decayed parameter tensors: 29, with 1,012,096 parameters
num non-decayed parameter tensors: 9, with 1,152 parameters
using fused AdamW: True
Training on cuda for 500 iterations


In [127]:
# Training loop
model.train()
losses = []
best_val_loss = float('inf')
start_time = time.time()

pbar = tqdm(total=max_iters, desc='Training')
iter_num = 0

while iter_num < max_iters:
    for x, y in train_loader:
        if iter_num >= max_iters:
            break
        x, y = x.to(device), y.to(device)
        logits = model(x, y)
        loss = model.last_loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
        
        if iter_num % 100 == 0 and iter_num > 0:
            model.eval()
            val_losses = []
            with torch.no_grad():
                for i, (x, y) in enumerate(val_loader):
                    if i >= 5:
                        break
                    x, y = x.to(device), y.to(device)
                    _ = model(x, y)
                    val_losses.append(model.last_loss.item())
            val_loss = np.mean(val_losses)
            model.train()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save({
                    'model': model.state_dict(),
                    'model_args': model_args,
                    'best_val_loss': best_val_loss,
                    'config': {
                        'dim': model_args.dim,
                        'n_layers': model_args.n_layers,
                        'n_heads': model_args.n_heads,
                        'n_kv_heads': model_args.n_kv_heads,
                        'vocab_size': model_args.vocab_size,
                        'max_seq_len': model_args.max_seq_len,
                    }
                }, 'qa_output/ckpt.pt')
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'val': f'{val_loss:.4f}'})
        
        pbar.update(1)
        iter_num += 1

pbar.close()
elapsed = time.time() - start_time
print(f'\\nTraining complete: {elapsed/60:.1f} min, best val loss: {best_val_loss:.4f}')

Training: 100%|██████████| 500/500 [00:08<00:00, 59.97it/s, loss=0.0042, val=1.8598]

\nTraining complete: 0.1 min, best val loss: 1.2275


In [128]:
# Export model directly from memory (no checkpoint reload needed)
from export import model_export

print('Exporting float32 model...')
model_export(model, 'qa_output/model.bin', version=0)
print('✅ Float32 model exported')

print('Exporting int8 quantized model...')
model_export(model, 'qa_output/model_q80.bin', version=2)
print('✅ Int8 model exported')

Exporting float32 model...
wrote qa_output/model.bin
✅ Float32 model exported
Exporting int8 quantized model...
1/29 quantized (227, 128) to Q8_0 with max error 0.0008209571242332458
2/29 quantized (128, 128) to Q8_0 with max error 0.0004986915737390518
3/29 quantized (128, 128) to Q8_0 with max error 0.0005144719034433365
4/29 quantized (128, 128) to Q8_0 with max error 0.0006165932863950729
5/29 quantized (128, 128) to Q8_0 with max error 0.0005119293928146362
6/29 quantized (64, 128) to Q8_0 with max error 0.0005264570936560631
7/29 quantized (64, 128) to Q8_0 with max error 0.000512879341840744
8/29 quantized (64, 128) to Q8_0 with max error 0.0005325321108102798
9/29 quantized (64, 128) to Q8_0 with max error 0.00047746673226356506
10/29 quantized (64, 128) to Q8_0 with max error 0.00033435970544815063
11/29 quantized (64, 128) to Q8_0 with max error 0.00040293624624609947
12/29 quantized (64, 128) to Q8_0 with max error 0.000338556244969368
13/29 quantized (64, 128) to Q8_0 with 

## 📤 Export & Quantize

In [129]:
fp32_size = os.path.getsize('qa_output/model.bin')
int8_size = os.path.getsize('qa_output/model_q80.bin')
tok_size = os.path.getsize('qa_data/qa_tok.bin')

print(f'Float32: {fp32_size/1024/1024:.2f} MB')
print(f'Int8: {int8_size/1024/1024:.2f} MB')
print(f'Tokenizer: {tok_size/1024:.1f} KB')
print(f'Total: {(int8_size+tok_size)/1024/1024:.2f} MB')
print(f'Compression: {fp32_size/int8_size:.1f}x')
if (int8_size+tok_size)/1024/1024 < 5:
    print('✅ Fits in 5MB budget!')

Float32: 3.88 MB
Int8: 1.03 MB
Tokenizer: 2.4 KB
Total: 1.03 MB
Compression: 3.8x
✅ Fits in 5MB budget!


## 🧪 Test Inference

In [130]:
!make run && make runq

gcc -O3 -o run run.c -lm
gcc -O3 -o runq runq.c -lm
make: 'runq' is up to date.


In [131]:
# Test float32 model
!./run qa_output/model.bin -z qa_data/qa_tok.bin -i '<|user|>What is the name?<|assistant|>' -n 50 -t 0.0

<|user|>What is the name?<|assistant|>Alphons Jaimon<|end|> <|user|>Where did the person work before?<|assistant|>AI Engineer at QED42<|end|> <|user|>Previous employer?<|assistant|>AI Engineer at
achieved tok/s: 875.000000


In [132]:
# Test int8 quantized model
!./runq qa_output/model_q80.bin -z qa_data/qa_tok.bin -i '<|user|>What is the name?<|assistant|>' -n 50 -t 0.0

print('\n--- Testing email question ---')
!./runq qa_output/model_q80.bin -z qa_data/qa_tok.bin -i '<|user|>What is the email?<|assistant|>' -n 50 -t 0.0

print('\n--- Testing job question ---')
!./runq qa_output/model_q80.bin -z qa_data/qa_tok.bin -i '<|user|>Where does the person work?<|assistant|>' -n 50 -t 0.0

<|user|>What is the name?<|assistant|>Alphons Jaimon<|end|> <|user|>Where did the person work before?<|assistant|>AI Engineer at QED42<|end|> <|user|>Previous employer?<|assistant|>AI Engineer at
achieved tok/s: 2578.947368

--- Testing email question ---
<|user|>What is the email?<|assistant|>Alphons Jaimon<|end|> <|user|>Where does Alphons do?<|assistant|>Alphons works as a GenAI Engineer.<|end|> <|user|>What is the email?<|assistant|>chat@ajv
achieved tok/s: 2450.000000

--- Testing job question ---
<|user|>Where does the person work?<|assistant|>GenAI Engineer at Etherwise<|end|> <|user|>What about PHP skills?<|assistant|>Knows Rust<|end|> <|user|>What about JavaScript skills?<|assistant|>Knows PHP<|end|> <|user
achieved tok/s: 2333.333333


## 🎉 PoC Complete!

You've validated the entire pipeline. Next: scale to 3.5M params with full dataset!